In [7]:
# 1. Cài đặt thư viện
!pip install transformers[torch] datasets sentencepiece Gradio -q

import os
import zipfile

# 2. Giải nén mô hình
def unzip_model(zip_path, extract_to):
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
            print(f"✅ Đã giải nén {zip_path} vào {extract_to}")
    else:
        print(f"❌ Không tìm thấy file: {zip_path}")

# Đường dẫn file zip của bạn
unzip_model('/content/bart_model-20260512T121256Z-3-001.zip', '/content/bart_model_extracted')
unzip_model('/content/t5_small_final-20260512T115703Z-3-001.zip', '/content/t5_model_extracted')

✅ Đã giải nén /content/bart_model-20260512T121256Z-3-001.zip vào /content/bart_model_extracted
✅ Đã giải nén /content/t5_small_final-20260512T115703Z-3-001.zip vào /content/t5_model_extracted


In [8]:
import gradio as gr
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, BartTokenizer, BartForConditionalGeneration

# --- Hàm hỗ trợ tìm đường dẫn chứa model ---
def find_model_dir(base_path):
    for root, dirs, files in os.walk(base_path):
        if "config.json" in files:
            return root
    return base_path

# Thiết lập đường dẫn sau khi giải nén
T5_PATH = find_model_dir('/content/t5_model_extracted')
BART_PATH = find_model_dir('/content/bart_model_extracted')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị: {device}")

# --- Tải Model ---
try:
    print(f"🔄 Đang tải T5 từ: {T5_PATH}")
    t5_tokenizer = T5Tokenizer.from_pretrained(T5_PATH, legacy=False)
    t5_model = T5ForConditionalGeneration.from_pretrained(T5_PATH).to(device)

    print(f"🔄 Đang tải BART từ: {BART_PATH}")
    bart_tokenizer = BartTokenizer.from_pretrained(BART_PATH)
    bart_model = BartForConditionalGeneration.from_pretrained(BART_PATH).to(device)

    print("✅ Tất cả mô hình đã sẵn sàng!")
except Exception as e:
    print(f"❌ Lỗi tải model: {e}")

# --- Hàm xử lý ---
def summarize_all(text):
    if not text.strip(): return "Nhập văn bản!", "Nhập văn bản!"

    try:
        # T5
        t5_in = t5_tokenizer("summarize: " + text, return_tensors="pt", max_length=512, truncation=True).to(device)
        t5_out = t5_model.generate(t5_in["input_ids"], max_length=150, min_length=40, num_beams=4)
        res_t5 = t5_tokenizer.decode(t5_out[0], skip_special_tokens=True)

        # BART
        bart_in = bart_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
        bart_out = bart_model.generate(bart_in["input_ids"], max_length=150, min_length=40, num_beams=4)
        res_bart = bart_tokenizer.decode(bart_out[0], skip_special_tokens=True)

        return res_t5, res_bart
    except Exception as e:
        return f"Lỗi: {str(e)}", f"Lỗi: {str(e)}"

# --- Giao diện ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 NLP Summary Tool: T5 vs BART")
    with gr.Row():
        input_txt = gr.Textbox(label="Văn bản gốc (English)", lines=10)
    btn = gr.Button("Tóm tắt ngay", variant="primary")
    with gr.Row():
        out_t5 = gr.Textbox(label="Kết quả T5-Small", interactive=False)
        out_bart = gr.Textbox(label="Kết quả BART-Base", interactive=False)

    btn.click(fn=summarize_all, inputs=input_txt, outputs=[out_t5, out_bart])

demo.launch(share=True)

Sử dụng thiết bị: cuda
🔄 Đang tải T5 từ: /content/t5_model_extracted/t5_small_final


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

🔄 Đang tải BART từ: /content/bart_model_extracted/bart_model


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

✅ Tất cả mô hình đã sẵn sàng!


/tmp/ipykernel_776/419628211.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c98c76b13463b22828.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
